### 5.2 RunnableLambda-임의의 함수를 Runnable로 만들기

예를 들어, LLM이 생성한 텍스트에 대해 소문자를 대문자로 변환하는 처리를 연결하는 Chain을 구현해보겠다.

In [1]:
# 먼저 prompt(ChatPromptTemplate), model(ChatOpenAI), output_parser(StrOutputParser)를 준비
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv("../.env")

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        ("human", "{input}")
    ]
)

model = ChatOpenAI(model="gpt-6-luna")
output_parser = StrOutputParser()

In [2]:
# 다음으로, 소문자를 대문자로 변환하는 함수를 구현하고, 이들을 Chain으로 연결하여 실행
from langchain_core.runnables import RunnableLambda

def upper(text: str) -> str:
    return text.upper()

chain = prompt | model | output_parser | RunnableLambda(upper)

ai_message = chain.invoke({"input": "Hello!"})
print(ai_message)

c:\Users\limux\Documents\1. Projects\llm-engineer-roadmap\02-rag-agents-langchain-langgraph\.venv\Lib\site-packages\langsmith\_internal\_addressing.py:104: LangSmithBetaWarning: Addressing runs with `langsmith.address` is in beta and is enabled per workspace. A workspace without it rejects the run, so the trace is lost rather than falling back to a project. The behavior may change without notice.
  _warn_once(


HELLO! HOW CAN I HELP YOU TODAY?


### chain 데코레이터를 사용한 RunnableLambda 구현

In [4]:
from langchain_core.runnables import chain

@chain
def upper(text: str) -> str:
    return text.upper()

chain = prompt | model | output_parser | upper
ai_message = chain.invoke({"input": "Hello!"})
print(ai_message)

HELLO! HOW CAN I HELP YOU TODAY?


### RunnableLambda 자동 변환

In [5]:
def upper(text: str) -> str:
    return text.upper()

chain = prompt | model | output_parser | upper

### Runnable 입력 타입과 출력 타입에 주의

In [6]:
# 예를 들어 다음 코드는 오류가 발생
def upper(text: str) -> str:
    return text.upper()

chain = prompt | model | upper
output = chain.invoke({"input": "Hello!"})

AttributeError: 'AIMessage' object has no attribute 'upper'

### 사용자 함수를 stream에 대응시키는 방법

In [7]:
from typing import Iterator

def upper(input_stream: Iterator[str]) -> Iterator[str]:
    for text in input_stream:
        yield text.upper()

chain = prompt | model | output_parser | upper

for chunk in chain.stream({"input": "Hello!"}):
    print(chunk, end="", flush=True)

HELLO! HOW CAN I HELP YOU TODAY?